# Baseline model

In [20]:
# Install required libraries
#!pip install datasets transformers

In [22]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from transformers.pipelines.pt_utils import KeyDataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Load and Prepare Dataset

In [8]:
# Load the dataset
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
print(f"Dataset splits: {dataset.keys()}")
print(f"Train set size: {len(dataset['train'])}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Dataset splits: dict_keys(['train'])
Train set size: 26872


In [9]:
dataset['train'] = dataset['train'].shuffle(seed=42)

train_testvalid = dataset['train'].train_test_split(test_size=0.3, seed=42)

test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Create final dataset dictionary
split_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']
})

print(f"Train set size: {len(split_dataset['train'])}")
print(f"Validation set size: {len(split_dataset['validation'])}")
print(f"Test set size: {len(split_dataset['test'])}")

Train set size: 18810
Validation set size: 4031
Test set size: 4031


## Explore Data

In [10]:
# Look at a sample
sample = split_dataset['train'][0]
print("Sample from training set:")
for key, value in sample.items():
    print(f"  {key}: {value}")

Sample from training set:
  flags: BIQ
  instruction: how to use the standard account
  category: ACCOUNT
  intent: switch_account
  response: Thank you for your interest in learning how to utilize our {{Account Type}} account. Allow me to guide you step by step to make the most of this option:

1. Login: Begin by accessing our platform using your credentials.
2. Navigation: Once logged in, familiarize yourself with the platform's interface.
3. Features: Explore the various features and tools available with the {{Account Type}} account.
4. Profile Customization: Personalize your profile by adding a profile picture and updating your information.
5. Account Settings: In the account settings section, you can configure preferences, update your password, and manage notification settings.
6. Usage Guidelines: Familiarize yourself with the terms of use and guidelines for the {{Account Type}} account.
7. Support and Resources: Should you have any further questions or require assistance, our su

In [11]:
# Get label information
labels = split_dataset['train']['intent']
unique_labels = set(labels)
label2id = {label: idx for idx, label in enumerate(sorted(unique_labels))}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Number of classes: {len(unique_labels)}")
print(f"\nLabel mapping:")
for label, idx in sorted(label2id.items(), key=lambda x: x[1]):
    print(f"  {idx}: {label}")

Number of classes: 27

Label mapping:
  0: cancel_order
  1: change_order
  2: change_shipping_address
  3: check_cancellation_fee
  4: check_invoice
  5: check_payment_methods
  6: check_refund_policy
  7: complaint
  8: contact_customer_service
  9: contact_human_agent
  10: create_account
  11: delete_account
  12: delivery_options
  13: delivery_period
  14: edit_account
  15: get_invoice
  16: get_refund
  17: newsletter_subscription
  18: payment_issue
  19: place_order
  20: recover_password
  21: registration_problems
  22: review
  23: set_up_shipping_address
  24: switch_account
  25: track_order
  26: track_refund


## Tokenization

In [12]:
# Load tokenizer
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(dataset):
    tokens = tokenizer(
        dataset['instruction'],
        padding='max_length',
        truncation=True,
        max_length=128
    )
    tokens['labels'] = [label2id[label] for label in dataset['intent']]
    return tokens

# Tokenize all datasets
tokenized_dataset = split_dataset.map(tokenize_function, batched=True)
print("Tokenization complete")
print(f"Tokenized train set size: {len(tokenized_dataset['train'])}")

Map:   0%|          | 0/4031 [00:00<?, ? examples/s]

Tokenization complete
Tokenized train set size: 18810


In [13]:
classifier = pipeline(
    "text-classification",
    model=model_name,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
classifier = pipeline(
    "text-classification",
    model=model_name
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
texts = split_dataset['test']['instruction'][:5]
predictions = classifier(texts)

for text, pred in zip(texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}\n")

Text: issues setting up a secondary shipping address
Prediction: {'label': 'LABEL_1', 'score': 0.5396633744239807}

Text: I havbe problems updating my delivery address
Prediction: {'label': 'LABEL_1', 'score': 0.5362477898597717}

Text: create another {{Account Category}} account for mum
Prediction: {'label': 'LABEL_1', 'score': 0.5633079409599304}

Text: I have got to retrieve my PIN code
Prediction: {'label': 'LABEL_1', 'score': 0.5438725352287292}

Text: where do i locate my invoices from {{Person Name}}
Prediction: {'label': 'LABEL_1', 'score': 0.5484386086463928}

